In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re



In [2]:
def parse_engine_params(row):
    s = str(row["engine"]) if pd.notna(row["engine"]) else ""

    # defaults
    engine_cc = np.nan
    horsepower_ag = np.nan
    power_kw = np.nan

    # electric: only AG and kW
    if row["is_electric"] == True:
        hp_match = re.search(r"(\d+)\s*AG", s)
        kw_match = re.search(r"\((\d+)\s*kW\)", s, flags=re.IGNORECASE)

        if hp_match:
            horsepower_ag = int(hp_match.group(1))
        if kw_match:
            power_kw = int(kw_match.group(1))

    # non-electric: cm³ + AG + kW
    else:
        cc_match = re.search(r"(\d+)\s*cm³", s)
        hp_match = re.search(r"(\d+)\s*AG", s)
        kw_match = re.search(r"\((\d+)\s*kW\)", s, flags=re.IGNORECASE)

        if cc_match:
            engine_cc = int(cc_match.group(1))
        if hp_match:
            horsepower_ag = int(hp_match.group(1))
        if kw_match:
            power_kw = int(kw_match.group(1))

    return pd.Series([engine_cc, horsepower_ag, power_kw])


In [3]:
df = pd.read_csv('autoplius_cars_04_23.csv')
df["first_registration_country"].unique()

array(['Vokietija', 'JAV', nan, 'Lietuva', 'Belgija', 'Olandija',
       'Švedija', 'Suomija', 'Danija', 'Italija', 'Prancūzija', 'Latvija',
       'Ispanija', 'Norvegija', 'Austrija', 'Kanada', 'Šveicarija',
       'Islandija', 'Estija', 'Slovėnija', 'Slovakija', 'Lenkija',
       'Liuksemburgas', 'Čekija', 'Japonija', 'Anglija', 'Airija'],
      dtype=object)

In [4]:
df["is_electric"] = df["ev_features"].notnull()

df[["engine_cc", "horsepower_ag", "power_kw"]] = df.apply(parse_engine_params, axis=1)

df["first_registration_year"] = pd.to_datetime(df["first_registration"], errors='coerce').dt.year
df["mileage"] = df["mileage_km"]
df["mileage_km"] = df["mileage"].apply(lambda x: float(x.replace(" ", "").replace("km", "")) if pd.notnull(x) else np.nan)
df["price_eur"] = df["price_eur"].apply(lambda x: float(x) if pd.notnull(x) else np.nan)

df["first_reg_abroad"] = df["first_registration_country"].apply(lambda x: x != "Lietuva" if pd.notnull(x) else False)

df.head()

,city_id,timestamp,URL,location,model,model_year,price_eur,first_registration,mileage_km,engine,...,other_features,safety_features,ev_features,is_electric,engine_cc,horsepower_ag,power_kw,first_registration_year,mileage,first_reg_abroad
0,1,2026-04-23T15:07:10.738222,NaN,"Vilnius, Lietuva",Honda Civic,2008,2380.0,2008-02,240564.0,"2204 cm³, 140 AG (103kW)",...,Serviso knygelė;Katalizatorius;Keli raktų komp...,Traukos kontrolės sistema;ESP;Tolimųjų šviesų ...,NaN,False,2204.0,140.0,103.0,2008.0,240 564 km,True
1,1,2026-04-23T15:07:11.028257,NaN,"Vilnius, Lietuva",BMW M550,2018,36000.0,2018-06,125894.0,"4400 cm³, 462 AG (340kW)",...,Parduodama lizingu;Serviso knygelė;Katalizator...,Traukos kontrolės sistema;ESP;Įkalnės stabdys;...,NaN,False,4400.0,462.0,340.0,2018.0,125 894 km,True
2,1,2026-04-23T15:07:11.206847,NaN,"Vilnius, Lietuva",Volkswagen Passat,1987,4000.0,1987-03,150000.0,"1800 cm³, 90 AG (66kW)",...,NaN,NaN,NaN,False,1800.0,90.0,66.0,1987.0,150 000 km,False
3,1,2026-04-23T15:07:11.453538,NaN,"Vilnius, Lietuva",BMW 325 Gran Turismo,2014,13500.0,2014-03,133000.0,"2500 cm³, 218 AG (160kW)",...,Parduodama lizingu;Katalizatorius;Keli raktų k...,Traukos kontrolės sistema;Aklosios zonos stebė...,NaN,False,2500.0,218.0,160.0,2014.0,133 000 km,False
4,1,2026-04-23T15:07:11.711701,NaN,"Vilnius, Lietuva",Volkswagen Golf,2011,3900.0,2011-05,307000.0,"1968 cm³, 140 AG (103kW)",...,Neeksploatuota Lietuvoje;Parduodama lizingu;Se...,Traukos kontrolės sistema;ESP;Įkalnės stabdys;...,NaN,False,1968.0,140.0,103.0,2011.0,307 000 km,True


In [5]:
df["manufacturer"] = df["model"].apply(lambda x: str(x).split(" ")[0] if pd.notnull(x) else np.nan)

df["manufacturer"].unique()

df[df["manufacturer"].isin(["Land", "Microcar", "DFSK", "Cadillac", "Infiniti", "GWM"])]["model"].unique()

array(['Land Rover Range Rover Sport', 'Cadillac BLS',
       'Land Rover Evoque', 'Cadillac DeVille', 'Cadillac XT6',
       'Infiniti Q50', 'Land Rover Discovery Sport',
       'Land Rover Range Rover', 'Land Rover Defender', 'Cadillac CTS',
       'Land Rover Discovery', 'Land Rover Range Rover Velar',
       'Infiniti Q60', 'Land Rover Freelander', 'Microcar F8C',
       'Microcar Mgo', 'Infiniti QX50', 'Infiniti QX4', 'Infiniti M30',
       'Microcar DUE', 'Cadillac CT6', 'Cadillac CT4', 'Cadillac ATS',
       'Infiniti Q30', 'Infiniti QX70', 'Infiniti EX30', 'Infiniti QX30',
       'Cadillac -kita-', 'Cadillac Escalade', 'Microcar M8'],
      dtype=object)

In [6]:
top_brands = df["manufacturer"].value_counts().head(10).index
df["manufacturer_grp"] = df["manufacturer"].where(
    df["manufacturer"].isin(top_brands), "Other"
)


In [7]:
df["first_registration_country_filled"] = df["first_registration_country"].fillna("Lietuva")

scand = ["Švedija", "Norvegija", "Suomija", "Danija", "Islandija"]
eu_other = ["Belgija", "Olandija", "Prancūzija", "Italija", "Austrija", "Čekija", "Lenkija", "Estija", "Latvija", "Slovakija", "Vengrija", "Rumunija", "Ispanija", "Liuksemburgas"]

def country_group(x):
    if pd.isna(x) or x == "Lietuva":
        return "Lithuania"
    elif x == "Vokietija":
        return "Germany"
    elif x in scand:
        return "Scandinavia"
    elif x in eu_other:
        return "Other_EU"
    else:
        return "Non_EU"

df["country_grp"] = df["first_registration_country_filled"].apply(country_group)

In [8]:
df.columns

ref_date = pd.Timestamp("2026-03-01")

df["ti_months"] = (
    (pd.to_datetime(df["technical_inspection_until"], errors="coerce") - ref_date)
    / pd.Timedelta(days=30.44)
).fillna(0)

df.head(5)

,city_id,timestamp,URL,location,model,model_year,price_eur,first_registration,mileage_km,engine,...,horsepower_ag,power_kw,first_registration_year,mileage,first_reg_abroad,manufacturer,manufacturer_grp,first_registration_country_filled,country_grp,ti_months
0,1,2026-04-23T15:07:10.738222,NaN,"Vilnius, Lietuva",Honda Civic,2008,2380.0,2008-02,240564.0,"2204 cm³, 140 AG (103kW)",...,140.0,103.0,2008.0,240 564 km,True,Honda,Other,Vokietija,Germany,17.017083
1,1,2026-04-23T15:07:11.028257,NaN,"Vilnius, Lietuva",BMW M550,2018,36000.0,2018-06,125894.0,"4400 cm³, 462 AG (340kW)",...,462.0,340.0,2018.0,125 894 km,True,BMW,BMW,JAV,Non_EU,15.998686
2,1,2026-04-23T15:07:11.206847,NaN,"Vilnius, Lietuva",Volkswagen Passat,1987,4000.0,1987-03,150000.0,"1800 cm³, 90 AG (66kW)",...,90.0,66.0,1987.0,150 000 km,False,Volkswagen,Volkswagen,Lietuva,Lithuania,24.014455
3,1,2026-04-23T15:07:11.453538,NaN,"Vilnius, Lietuva",BMW 325 Gran Turismo,2014,13500.0,2014-03,133000.0,"2500 cm³, 218 AG (160kW)",...,218.0,160.0,2014.0,133 000 km,False,BMW,BMW,Lietuva,Lithuania,0.000000
4,1,2026-04-23T15:07:11.711701,NaN,"Vilnius, Lietuva",Volkswagen Golf,2011,3900.0,2011-05,307000.0,"1968 cm³, 140 AG (103kW)",...,140.0,103.0,2011.0,307 000 km,True,Volkswagen,Volkswagen,Vokietija,Germany,0.000000


In [9]:


#df[~pd.isna(df["defects"])].head(10)

df["has_defects"] = df["defects"].apply(lambda x: True if pd.notna(x) else False)


df["fuel_type"].unique()

array(['Dyzelinas', 'Benzinas', 'Benzinas / dujos',
       'Benzinas / elektra / dujos'], dtype=object)

In [10]:

df = df[df["city_id"] != 3]
df_cleaned = df[["first_registration_year", "mileage_km", "engine_cc", "horsepower_ag", "power_kw", "is_electric", "model", "price_eur", "location", "first_reg_abroad", "country_grp", "manufacturer_grp", "ti_months", "has_defects", "fuel_type", "ad_id"]]

df_cleaned.to_csv("autoplius_cars_04_23_c.csv", index=False)